# Aula 04 · SQL intermediário no Databricks

## O que é
SQL é a forma de **fazer perguntas aos dados**. Nesta aula vamos de perguntas simples a um relatório completo usando:
`JOIN`, `GROUP BY` + `HAVING`, `CASE WHEN`, `CTE` (`WITH`) e **funções de janela**.

As consultas podem ser feitas aqui, em células `%sql`, ou no **SQL Editor** (menu lateral). Experimente as duas!

## Por que importa
Boa parte das transformações e validações de dados é escrita em SQL, e **tarefas SQL** são um dos tipos de tarefa dos jobs.

## Como aplicar
Usamos as tabelas criadas na aula 03: `growdev.nivelamento.clientes` e `growdev.nivelamento.emprestimos`.

In [ ]:
%sql
SELECT e.loan_id, c.full_name, c.state, e.loan_category, e.valor, e.status
FROM growdev.nivelamento.emprestimos AS e
JOIN growdev.nivelamento.clientes AS c
  ON e.customer_id = c.customer_id
LIMIT 10

In [ ]:
%sql
SELECT c.state, COUNT(*) AS qtd_emprestimos, ROUND(AVG(e.valor), 2) AS valor_medio
FROM growdev.nivelamento.emprestimos AS e
JOIN growdev.nivelamento.clientes AS c ON e.customer_id = c.customer_id
GROUP BY c.state
HAVING COUNT(*) > 1000
ORDER BY qtd_emprestimos DESC

In [ ]:
%sql
SELECT
  CASE
    WHEN valor < 10000 THEN '1. Até 10 mil'
    WHEN valor < 50000 THEN '2. De 10 a 50 mil'
    ELSE '3. Acima de 50 mil'
  END AS faixa_valor,
  COUNT(*) AS qtd_emprestimos
FROM growdev.nivelamento.emprestimos
WHERE valor IS NOT NULL
GROUP BY faixa_valor
ORDER BY faixa_valor

In [ ]:
%sql
WITH emprestimos_por_categoria AS (
  SELECT
    loan_category,
    COUNT(*) AS total,
    SUM(CASE WHEN inadimplente THEN 1 ELSE 0 END) AS inadimplentes
  FROM growdev.nivelamento.emprestimos
  GROUP BY loan_category
)
SELECT loan_category, total, inadimplentes,
       ROUND(100 * inadimplentes / total, 2) AS taxa_inadimplencia_pct
FROM emprestimos_por_categoria
ORDER BY taxa_inadimplencia_pct DESC

In [ ]:
%sql
SELECT c.state, c.full_name, SUM(e.valor) AS valor_total,
       RANK() OVER (PARTITION BY c.state ORDER BY SUM(e.valor) DESC) AS posicao_no_estado
FROM growdev.nivelamento.emprestimos AS e
JOIN growdev.nivelamento.clientes AS c ON e.customer_id = c.customer_id
GROUP BY c.state, c.full_name
QUALIFY posicao_no_estado <= 3
ORDER BY c.state, posicao_no_estado

### 6. Consulta de relatório
Juntando tudo: um relatório por estado com volume, taxa de inadimplência e posição no ranking nacional.

In [ ]:
%sql
WITH base AS (
  SELECT c.state, e.valor, e.inadimplente
  FROM growdev.nivelamento.emprestimos AS e
  JOIN growdev.nivelamento.clientes AS c ON e.customer_id = c.customer_id
),
por_estado AS (
  SELECT
    state,
    COUNT(*) AS qtd_emprestimos,
    ROUND(SUM(valor), 2) AS valor_total,
    ROUND(100 * AVG(CASE WHEN inadimplente THEN 1 ELSE 0 END), 2) AS taxa_inadimplencia_pct
  FROM base
  GROUP BY state
)
SELECT *,
       RANK() OVER (ORDER BY valor_total DESC) AS ranking_valor,
       CASE WHEN taxa_inadimplencia_pct > 10 THEN 'Atenção' ELSE 'Normal' END AS situacao
FROM por_estado
ORDER BY ranking_valor

## Checklist da aula
- [ ] Juntei tabelas com `JOIN`
- [ ] Filtrei grupos com `HAVING`
- [ ] Criei categorias com `CASE WHEN`
- [ ] Organizei a consulta em etapas com `WITH` (CTE)
- [ ] Usei uma função de janela (`RANK() OVER`) e filtrei com `QUALIFY`